In [6]:
import os
from pyspark.sql import SparkSession
ss=SparkSession.builder.appName('app_for_struct_stream').getOrCreate()


# In[2]:


ss


# In[3]:


ss.conf.set("spark.sql.streaming.schemaInference", True)
read_stream_df =    ss.readStream.format("csv")\
                    .options(header = True,delimiter = ",", recursiveFileLookup = True )\
                    .load("file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_csv_files/")


# In[4]:


from pyspark.sql.functions import udf
@udf
def func_date_time():
    return str(datetime.now())


# In[5]:


from datetime import datetime,timedelta
transform_df = read_stream_df.withColumn("Read_time",func_date_time())


# In[6]:


from pyspark.sql.types import *
from pyspark.sql.functions import *
transform_df_change_type = transform_df.withColumn("col_sales_int_type", col("col_sales").cast(IntegerType()))\
                                        .withColumn("Read_time_type_tmstmp", col("Read_time").cast(TimestampType()))\
                                        .withColumn("my_time_type_tmstmp", col("my_time").cast(TimestampType()))


# In[7]:


transform_df_change_type.printSchema()

root
 |-- col_company: string (nullable = true)
 |-- col_product: string (nullable = true)
 |-- col_number_of_items: string (nullable = true)
 |-- col_sales: string (nullable = true)
 |-- my_time: string (nullable = true)
 |-- Read_time: string (nullable = true)
 |-- col_sales_int_type: integer (nullable = true)
 |-- Read_time_type_tmstmp: timestamp (nullable = true)
 |-- my_time_type_tmstmp: timestamp (nullable = true)



In [15]:
writing_df_append =    transform_df_change_type.writeStream\
                .format("csv")\
                .options(header = True,delimiter = ",")\
                .option("path", "file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_output_of_ss/")\
                .option("checkpointLocation","file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_checkpointing_location/")\
                .outputMode("append")\
                .start()
writing_df_append.awaitTermination()

25/06/03 16:45:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/yvm/Documents/y_python_envs/y_env_for_spark/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yvm/Documents/y_python_envs/y_env_for_spark/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [9]:
writing_df_append =    transform_df_change_type.withWatermark("my_time_type_tmstmp", "0 seconds")\
                .groupBy(
                    window(col("my_time_type_tmstmp"), "10 seconds"),
                    col("col_company")
                ) \
                .agg(count("*").alias("event_count")) \
                .writeStream\
                .format("parquet")\
                .options(header = True,delimiter = ",")\
                .option("path", "file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_output_of_ss/")\
                .option("checkpointLocation","file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_checkpointing_location/")\
                .outputMode("append")\
                .start()
writing_df_append.awaitTermination()

25/06/03 17:30:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/home/yvm/Documents/y_python_envs/y_env_for_spark/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yvm/Documents/y_python_envs/y_env_for_spark/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
spark.read.format('parquet').load("file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_output_of_ss/").show()

In [ ]:
dd.show()


25/07/03 16:37:37 WARN CheckpointFileManager: Failed to rename temp file file:/home/yvm/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_checkpointing_location/sources/0/.2.98edc9fa-5df8-41d3-97ef-4cc8ee936130.tmp to file:/home/yvm/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_checkpointing_location/sources/0/2 because file exists
org.apache.hadoop.fs.FileAlreadyExistsException: rename destination file:/home/yvm/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_checkpointing_location/sources/0/2 already exists.
	at org.apache.hadoop.fs.FileSystem.rename(FileSystem.java:1600)
	at org.apache.hadoop.fs.DelegateToFileSystem.renameInternal(DelegateToFileSystem.java:206)
	at org.apache.hadoop.fs.AbstractFileSystem.renameInternal(AbstractFileSystem.java:790)
	at org.apache.hadoop.fs.AbstractFileSystem.rename(AbstractFileSystem.java:720)
	at org.apache.hadoop.fs.ChecksumFs.renameInternal(ChecksumFs.java:489)
	at org.apache.hadoop.fs.Abstract